# Topshe trainer (Qwen 3.5 4B)

Used for the first training run **and** every weekly retrain (the GitHub pipeline pushes it here automatically).

- Accelerator: **GPU T4 x2** (one GPU is used). Internet: **On**.
- If the `topshe-data` dataset is attached: your real usage, corrections, past messages and last week's scores are used.
- Outputs: `topshe_brain.gguf` (phone model), `prompt_template.json`, `eval.json` (test scores),
  `chat_distill.jsonl` (reused next week), `lora/` (backup).


In [ ]:
# 1. Setup (pinned versions)
import os, time, json, glob, random, shutil, subprocess, hashlib, math
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
T0 = time.time()
!pip install -q "unsloth==2026.11.9" "unsloth_zoo==2026.11.9" "transformers>=5.2.0,<=5.17.0"
!nvidia-smi --query-gpu=name,memory.total --format=csv
OUT = "/kaggle/working"
TMP = "/tmp/topshe"
os.makedirs(TMP, exist_ok=True)

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


In [ ]:
# 2. Settings
BASE_MODEL = "unsloth/Qwen3.5-4B"
LLAMA_CPP_TAG = "b12602"
MAX_LEN = 1024
TRAIN_HOURS_LIMIT = 3.5        # training stops safely after this long
N_DISTILL = 600                # general chat examples (made once, reused every week)
N_STYLE = 1500                 # your past messages used per run


In [ ]:
# 3. Training data generator (training/make_dataset_v3.py)
import json
import random
import re

random.seed(42)

"""
Topshe training data v3.

Shared by the first training run and every weekly retrain. The Android app uses the
SAME system prompts (Brain.kt), so keep them in sync:
  - CMD_SYSTEM  + optional facts block  -> one compact JSON action
  - CHAT_SYSTEM + optional facts block  -> a short spoken answer
"""

CMD_SYSTEM = (
    "You are Topshe, a phone assistant. Turn the user's request into ONE JSON object. "
    "Actions: set_alarm, set_timer, set_reminder, open_app, call, message, tasker, media, "
    "remember, forget, stop, chat. Use chat for questions or small talk. Reply with JSON only."
)
CHAT_SYSTEM = (
    "You are Topshe, a friendly assistant. Answer in 1-3 short, simple sentences. "
    "Talk to Annay plainly, like an adult. No lectures or disclaimers."
)
FACTS_HEADER = "What you know about Annay:"


def with_facts(system, facts):
    """System prompt plus the memory block, exactly as the app builds it."""
    if not facts:
        return system
    return system + "\n\n" + FACTS_HEADER + "\n" + "\n".join("- " + f for f in facts)


def dumps(obj):
    """Compact JSON (fewer tokens = faster replies on the phone)."""
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))

# Add your own apps and contacts here, then re-run
APPS = ["WhatsApp", "YouTube", "Spotify", "Chrome", "Gmail", "Maps", "Camera", "Instagram",
        "Telegram", "Settings", "Calculator", "Calendar", "Kindle", "Zomato", "Swiggy", "Uber",
        "Ola", "Google Pay", "PhonePe", "Notes", "Files", "Clock", "Photos", "Zerodha Kite",
        "Netflix", "Hotstar", "Drive", "Meet", "Zoom", "Slack", "LinkedIn", "Twitter",
        "Blinkit", "Zepto", "Amazon", "Flipkart", "Paytm", "Outlook", "Teams", "Discord"]
CONTACTS = ["Maa", "Baba", "Tanmay", "Rahul", "Priya", "Didi", "Dada", "Arjun", "Sneha",
            "Professor Sen", "Ankit", "Riya", "Mama", "Kaku", "Shreya", "Sourav", "Ritwik",
            "Pishi", "Jethu", "Anirban", "Debjani", "Rohan", "Ishita", "Sayan", "Moumita"]
MSG_TEXTS = ["I'll be late", "reached home", "call me when free", "on my way",
             "running 10 minutes late", "can we meet tomorrow", "sent the file",
             "happy birthday", "I'm in class right now", "dinner at 9?", "the draft is ready",
             "I'm taking the shuttle", "will call you later", "check your email",
             "ok see you soon", "I'm in a meeting"]
TASKS = ["wifi", "bluetooth", "silent mode", "dnd", "torch", "battery saver", "hotspot",
         "dark mode", "study mode", "mobile data", "location", "airplane mode"]
REMIND_TEXTS = ["submit the assignment", "call Maa", "take the shuttle", "RA meeting",
                "buy milk", "pay the electricity bill", "check email", "drink water",
                "start the Stata run", "read the paper", "take my medicine", "book the cab",
                "charge the laptop", "reply to the professor", "go to the gym"]
CHAT_Q = ["What is the capital of Japan?", "Who wrote Sonar Kella?", "How far is the moon?",
          "Tell me a joke", "How are you?", "What is 17 times 23?", "Who is Feluda?",
          "What should I cook tonight?", "How many litres in a gallon?", "Give me a fun fact",
          "Translate thank you to Bengali", "Thank you Topshe", "Suggest a good book",
          "Why is the sky blue?", "What's a p-value?", "Good morning", "What can you do?",
          "Who are you?", "How many days in a leap year?", "What's 15 percent of 240?"]
TOPICS = ["black holes", "the Mughal empire", "Satyajit Ray", "photosynthesis", "inflation",
          "machine learning", "the stock market", "Durga Puja", "the Bengal famine", "cricket",
          "climate change", "the RBI", "vaccines", "the Taj Mahal", "Rabindranath Tagore",
          "game theory", "the French Revolution", "monsoons", "GST", "quantum computing",
          "the Sundarbans", "regression", "Netaji Subhas Chandra Bose", "the IPL", "DNA",
          "instrumental variables", "mutual funds", "the Indian constitution", "volcanoes"]
CHAT_TMPL = ["what is {x}", "tell me about {x}", "explain {x} simply", "who started {x}",
             "why is {x} important", "give me a fact about {x}", "what do you know about {x}",
             "{x} ke baare mein batao", "{x} niye kichu bolo", "summarise {x} in two lines",
             "how does {x} work", "is {x} a good thing"]


def fmt_time(h, m):
    ampm = "am" if h < 12 else "pm"
    h12 = h % 12 or 12
    opts = [f"{h12}:{m:02d} {ampm}", f"{h12}:{m:02d}{ampm}", f"{h12} {m:02d} {ampm}"]
    if m == 0:
        opts += [f"{h12} {ampm}", f"{h12} o'clock " + ("in the morning" if h < 12 else "in the evening")]
    if m == 30:
        opts.append(f"half past {h12} {ampm}")
    if m == 15:
        opts.append(f"quarter past {h12} {ampm}")
    return random.choice(opts)


def ex_alarm():
    h = random.randint(0, 23)
    m = random.choice([0, 0, 0, 15, 30, 30, 45, random.randint(0, 59)])
    tmpl = random.choice([
        "wake me up at {t}", "set an alarm for {t}", "alarm at {t}", "set alarm {t} tomorrow",
        "I need to get up at {t} tomorrow", "kal subah {t} baje alarm laga do", "{t} er alarm dao",
        "kal {t} e alarm dao", "alarm for {t} today", "put an alarm at {t}",
        "set a {t} alarm", "{t} pe alarm set karo",
    ])
    day = "tomorrow" if ("tomorrow" in tmpl or "kal" in tmpl) else ("today" if "today" in tmpl else "next")
    return tmpl.format(t=fmt_time(h, m)), {"action": "set_alarm", "hour": h, "minute": m, "day": day}


def ex_timer():
    unit = random.choice(["seconds", "minutes", "minutes", "minutes", "hours"])
    n = random.randint(1, 3) if unit == "hours" else random.randint(1, 90)
    word = unit[:-1] if n == 1 else unit
    secs = n * {"seconds": 1, "minutes": 60, "hours": 3600}[unit]
    text = random.choice([
        "set a timer for {n} {u}", "timer {n} {u}", "start a {n} {u} timer", "{n} {u} ka timer lagao",
        "count down {n} {u}", "{n} {u} er timer dao", "time {n} {u}",
    ]).format(n=n, u=word)
    return text, {"action": "set_timer", "seconds": secs}


def ex_reminder():
    what = random.choice(REMIND_TEXTS)
    if random.random() < 0.5:
        n = random.choice([5, 10, 15, 20, 30, 45, 60, 90, 120])
        text = random.choice(["remind me to {w} in {n} minutes", "in {n} minutes remind me to {w}",
                              "{n} minute baad yaad dilana {w}", "remind me in {n} minutes to {w}"]).format(w=what, n=n)
        return text, {"action": "set_reminder", "text": what, "in_minutes": n}
    h, m = random.randint(6, 23), random.choice([0, 15, 30, 45])
    day = random.choice(["today", "tomorrow"])
    text = random.choice(["remind me to {w} at {t} {d}", "{d} at {t} remind me to {w}",
                          "set a reminder to {w} at {t} {d}", "remind me {d} at {t} to {w}"]).format(w=what, t=fmt_time(h, m), d=day)
    return text, {"action": "set_reminder", "text": what, "hour": h, "minute": m, "day": day}


def ex_open():
    app = random.choice(APPS)
    text = random.choice(["open {a}", "launch {a}", "start {a}", "{a} kholo", "go to {a}",
                          "can you open {a}", "{a} khulo", "open up {a}", "take me to {a}"]).format(a=app)
    return text, {"action": "open_app", "app": app}


def ex_call():
    c = random.choice(CONTACTS)
    text = random.choice(["call {c}", "phone {c}", "ring {c}", "{c} ko call karo", "{c} ke phone koro",
                          "give {c} a call", "dial {c}", "call {c} now", "make a call to {c}"]).format(c=c)
    return text, {"action": "call", "contact": c}


def ex_message():
    c, msg = random.choice(CONTACTS), random.choice(MSG_TEXTS)
    if random.random() < 0.3:
        app = "SMS"
        text = random.choice(["send {c} an SMS saying {m}", "SMS {c} {m}", "text {c} by SMS that {m}"]).format(c=c, m=msg)
    elif random.random() < 0.5:
        app = "WhatsApp"
        text = random.choice(["whatsapp {c} {m}", "send {c} a whatsapp saying {m}",
                              "tell {c} on whatsapp that {m}"]).format(c=c, m=msg)
    else:
        app = "WhatsApp"  # default when no app is named
        text = random.choice(["message {c} {m}", "text {c} saying {m}", "tell {c} {m}",
                              "{c} ko message bhejo {m}", "send a message to {c} that {m}"]).format(c=c, m=msg)
    return text, {"action": "message", "app": app, "contact": c, "text": msg}


def ex_tasker():
    t, state = random.choice(TASKS), random.choice(["on", "off"])
    text = random.choice(["turn {s} {t}", "turn {t} {s}", "switch {s} {t}", "switch {t} {s}",
                          "{t} {s}", "{t} {s} karo", "{t} {s} koro", "put {t} {s}"]).format(t=t, s=state)
    return text, {"action": "tasker", "task": f"{t} {state}"}


def ex_stop():
    text = random.choice(["stop listening", "go to sleep", "topshe off", "turn yourself off",
                          "that's all", "bye topshe", "stop", "chup karo", "thamo", "you can sleep now"])
    return text, {"action": "stop"}


def ex_chat():
    q = random.choice(CHAT_Q) if random.random() < 0.3 else random.choice(CHAT_TMPL).format(x=random.choice(TOPICS))
    return q, {"action": "chat", "query": q}


SONGS = ["Tum Hi Ho", "Blinding Lights", "Kesariya", "Shape of You", "Apna Bana Le", "Levitating",
         "Amar Shonar Bangla", "Ek Mulaqat", "Bohemian Rhapsody", "Channa Mereya", "Pasoori",
         "Hotel California", "Ami Je Tomar", "Kun Faya Kun", "Perfect", "Tere Bina", "Starboy"]
ARTISTS = ["Arijit Singh", "The Weeknd", "Anuv Jain", "Prateek Kuhad", "Coldplay", "Taylor Swift",
           "Shreya Ghoshal", "A R Rahman", "Anupam Roy", "Ed Sheeran", "Lata Mangeshkar", "Kishore Kumar"]


def ex_media():
    r = random.random()
    if r < 0.35:
        q = random.choice(SONGS + ARTISTS + [f"{s} by {a}" for s, a in zip(SONGS, ARTISTS)])
        text = random.choice(["play {q}", "play {q} on spotify", "put on {q}", "{q} chalao", "{q} bajao",
                              "i want to hear {q}", "can you play {q}", "{q} er gaan chalao", "{q} chalao na",
                              "spotify te {q} chalao"]).format(q=q)
        return text, {"action": "media", "op": "play", "query": q, "source": "spotify"}
    if r < 0.5:
        q = random.choice(["lofi beats", "the news", "a cooking video", "IPL highlights", "standup comedy",
                           "Satyajit Ray interview", "study music", random.choice(SONGS)])
        text = random.choice(["play {q} on youtube", "youtube pe {q} chalao", "open {q} on youtube",
                              "find {q} on youtube"]).format(q=q)
        return text, {"action": "media", "op": "play", "query": q, "source": "youtube"}
    op, phrases = random.choice([
        ("play", ["play some music", "play music", "resume the music", "music chalao", "gaana chalao", "gaan chalao"]),
        ("pause", ["pause the music", "pause it", "stop the song", "gaana band karo"]),
        ("next", ["next song", "skip this", "play the next one", "agla gaana"]),
        ("previous", ["previous song", "go back one song", "play the last song again"]),
        ("volume_up", ["turn it up", "louder please", "increase the volume", "awaaz badhao"]),
        ("volume_down", ["turn it down", "lower the volume", "awaaz kam karo"]),
        ("mute", ["mute", "mute the music"]),
    ])
    out = {"action": "media", "op": op}
    if op == "play":
        out.update({"query": "", "source": "spotify"})
    return random.choice(phrases), out


FACTS = ["my shuttle leaves at 8:10 in the morning", "my RA meeting is every Friday at 5 pm",
         "Rahul means Rahul Sen", "my sister is called Riya", "I like my coffee without sugar",
         "my Kaggle username is annayde", "Tanmay is my term paper partner", "my mother's birthday is on 12 March",
         "I take the 9 pm metro home", "my laptop password hint is the dog's name",
         "my favourite singer is Arijit Singh", "my gym days are Monday, Wednesday and Friday",
         "Professor Sen's office is in the AC 04 building", "my landlord is called Mr Gupta",
         "I'm vegetarian on Tuesdays", "my car is parked in slot B12"]


def ex_remember():
    f = random.choice(FACTS)
    if random.random() < 0.75:
        text = random.choice(["remember that {f}", "remember {f}", "note that {f}", "keep in mind {f}",
                              "yaad rakhna {f}", "save this: {f}", "don't forget that {f}"]).format(f=f)
        return text, {"action": "remember", "fact": f}
    text = random.choice(["forget that {f}", "forget about {f}", "you can forget {f}", "delete the note that {f}"]).format(f=f)
    return text, {"action": "forget", "fact": f}


# Commands that need the memory block to resolve ("call my sister" -> Riya)
MEMORY_CMDS = [
    (["my sister is called Riya"], ["call my sister", "phone my sister", "give my sister a call"], {"action": "call", "contact": "Riya"}),
    (["Rahul means Rahul Sen"], ["call Rahul", "ring Rahul"], {"action": "call", "contact": "Rahul Sen"}),
    (["my landlord is called Mr Gupta"], ["call my landlord", "phone the landlord"], {"action": "call", "contact": "Mr Gupta"}),
    (["Tanmay is my term paper partner"], ["message my term paper partner that I sent the draft"],
     {"action": "message", "app": "WhatsApp", "contact": "Tanmay", "text": "I sent the draft"}),
    (["my favourite singer is Arijit Singh"], ["play my favourite singer", "play songs by my favourite singer"],
     {"action": "media", "op": "play", "query": "Arijit Singh", "source": "spotify"}),
    (["my shuttle leaves at 8:10 in the morning"], ["remind me 15 minutes before my shuttle"],
     {"action": "set_reminder", "text": "catch the shuttle", "hour": 7, "minute": 55, "day": "next"}),
]

# Chat that should use the memory block
MEMORY_CHATS = [
    ("my shuttle leaves at 8:10 in the morning", ["when is my shuttle?", "what time is my shuttle"], "Your shuttle leaves at 8:10 in the morning."),
    ("my RA meeting is every Friday at 5 pm", ["when is my RA meeting?", "when's the RA meeting"], "Your RA meeting is every Friday at 5 pm."),
    ("my sister is called Riya", ["what's my sister's name?", "who is my sister"], "Your sister is Riya."),
    ("my mother's birthday is on 12 March", ["when is my mom's birthday?", "maa ka birthday kab hai"], "Your mother's birthday is on 12 March."),
    ("my car is parked in slot B12", ["where did I park?", "where's my car"], "Your car is in slot B12."),
    ("my gym days are Monday, Wednesday and Friday", ["which days do I go to the gym?"], "You go to the gym on Monday, Wednesday and Friday."),
    ("Professor Sen's office is in the AC 04 building", ["where is Professor Sen's office?"], "Professor Sen's office is in the AC 04 building."),
]


# Facts that change who a name refers to: never shown with generic commands (would contradict labels)
ALIAS_FACTS = {f for must, _, _ in MEMORY_CMDS for f in must}


# Who Topshe is (written for Topshe; answers kept short, like Annay wants)
IDENTITY_CHATS = [
    (["who are you", "what are you", "tumi ke", "introduce yourself"],
     "I'm Topshe, your assistant. I play music, set alarms and reminders, call or message people, open apps and answer questions."),
    (["what can you do", "what all can you do", "help"],
     "I can play music, set alarms, timers and reminders, call or message people, open apps, run Tasker tasks, remember things for you and answer questions."),
    (["who made you", "who built you"], "Annay built me."),
    (["are you listening", "you there", "topshe are you there"], "Yes, I'm here."),
    (["thank you", "thanks topshe", "thanks"], "Anytime."),
    (["good night topshe", "good night"], "Good night, Annay."),
]

# Held-out memory checks (facts NEVER used in training): does Topshe use what it was told?
EVAL_MEMORY_CHATS = [
    ("my dentist appointment is on Thursday at 4 pm", "when is my dentist appointment?", "4"),
    ("my bike is a blue Hero Splendor", "what bike do I have?", "splendor"),
    ("my favourite food is kosha mangsho", "what's my favourite food?", "kosha"),
    ("Riju is my cousin", "who is Riju?", "cousin"),
    ("my flight to Kolkata is on 18 December", "when is my flight?", "18"),
    ("my locker code hint is my roll number", "what's my locker code hint?", "roll"),
    ("my thesis advisor is Professor Mehta", "who is my thesis advisor?", "mehta"),
    ("I keep my passport in the blue folder", "where is my passport?", "blue"),
]
EVAL_MEMORY_CMDS = [
    ("my cousin is called Riju", "call my cousin", {"action": "call", "contact": "Riju"}),
    ("my advisor is Professor Mehta", "message my advisor that I'll be late",
     {"action": "message", "app": "WhatsApp", "contact": "Professor Mehta", "text": "I'll be late"}),
    ("my favourite band is Coldplay", "play my favourite band",
     {"action": "media", "op": "play", "query": "Coldplay", "source": "spotify"}),
]


def random_facts(k_max=6, must=None):
    pool = [f for f in FACTS if f not in (must or []) and f not in ALIAS_FACTS]
    facts = random.sample(pool, random.randint(0, k_max)) + list(must or [])
    random.shuffle(facts)
    return facts


GENERATORS = [(ex_alarm, 600), (ex_timer, 350), (ex_reminder, 600), (ex_open, 450), (ex_call, 400),
              (ex_message, 600), (ex_tasker, 350), (ex_stop, 100), (ex_chat, 600), (ex_media, 700),
              (ex_remember, 300)]


def noisy(text):
    """Make text look like speech-to-text output."""
    if random.random() < 0.15:
        text = random.choice(["topshe ", "hey topshe ", "please ", "umm ", "ok "]) + text
    if random.random() < 0.1:
        text += random.choice([" please", " thanks", " now"])
    text = text.replace("?", "") if random.random() < 0.5 else text
    return text.lower() if random.random() < 0.6 else text


def row(system, user, assistant, kind):
    return {"system": system, "user": user, "assistant": assistant, "kind": kind}


def make_rows(seed=42):
    """Synthetic training rows: dicts with system / user / assistant / kind."""
    random.seed(seed)
    rows, seen = [], set()
    for gen, n in GENERATORS:
        made = tries = 0
        while made < n and tries < n * 30:
            tries += 1
            text, out = gen()
            text = " ".join(noisy(text).split())
            if text in seen:
                continue
            seen.add(text)
            # 40% of commands see a memory block (the app always sends it when facts exist)
            facts = random_facts() if random.random() < 0.4 else []
            rows.append(row(with_facts(CMD_SYSTEM, facts), text, dumps(out), "command"))
            made += 1
    for _ in range(15):
        for must, phrasings, out in MEMORY_CMDS:
            for p in phrasings:
                rows.append(row(with_facts(CMD_SYSTEM, random_facts(5, must)), noisy(p), dumps(out), "memory_command"))
    for _ in range(10):
        for fact, questions, answer in MEMORY_CHATS:
            for q in questions:
                rows.append(row(with_facts(CHAT_SYSTEM, random_facts(5, [fact])), noisy(q), answer, "memory_chat"))
    for _ in range(8):
        for qs, answer in IDENTITY_CHATS:
            for q in qs:
                rows.append(row(with_facts(CHAT_SYSTEM, random_facts(4) if random.random() < 0.4 else []),
                                noisy(q), answer, "identity"))
    random.shuffle(rows)
    return rows


def eval_memory_rows():
    """Held-out memory checks (never trained on)."""
    chats = [row(with_facts(CHAT_SYSTEM, random_facts(4, [f])), q, key, "eval_memory_chat") for f, q, key in EVAL_MEMORY_CHATS]
    cmds = [row(with_facts(CMD_SYSTEM, random_facts(4, [f])), q, dumps(out), "eval_memory_cmd") for f, q, out in EVAL_MEMORY_CMDS]
    return chats, cmds


def user_rows(interactions, facts, now_ts=None):
    """
    Turns the phone's interaction log into training rows. Only trusted examples:
      - ok / confirmed commands (Topshe did it and you didn't object)
      - corrections ("no, I meant ...") -> the corrected action
    Cancelled actions and Topshe's own chat answers are never trained on.

    Each row gets a weight (how many copies go into training):
      correction 5, confirmed 3, ok 2;  x1.5 if from the last 14 days.
    """
    import time as _time
    now_ts = now_ts or _time.time()
    final = {}   # last word wins: a later correction or cancel overrides an earlier "ok"
    for it in interactions:
        heard = (it.get("heard") or "").strip()
        if heard:
            final[heard.lower()] = (heard, it)
    out = []
    for heard, it in final.values():
        outcome = it.get("outcome")
        action = it.get("corrected_to") if outcome == "corrected" else it.get("action")
        if outcome not in ("ok", "confirmed", "corrected") or not isinstance(action, dict):
            continue
        if action.get("action") == "chat":
            continue
        w = {"corrected": 5, "confirmed": 3, "ok": 2}[outcome]
        if now_ts - float(it.get("ts") or 0) < 14 * 86400:
            w *= 1.5
        r = row(with_facts(CMD_SYSTEM, facts), heard, dumps(action), "user")
        r["weight"] = max(1, round(w))
        out.append(r)
    return out


COMMAND_START = re.compile(r"^(hey |ok )?(topshe )?(play|pause|call|ring|phone|message|text|whatsapp|remind|set|wake|timer|"
                           r"alarm|open|launch|turn|switch|remember|forget|stop|next|skip|volume|mute)\b", re.I)


# Phone-command phrases anywhere in a message: skip it (it might really be a command)
COMMAND_ANYWHERE = re.compile(r"\b(remind me|set (an |a )?(alarm|timer|reminder)|wake me|call (my|maa|baba|him|her)|"
                              r"play (some|the|a|my)|open (the )?app|whatsapp|send (a )?message|turn (on|off)|volume)\b", re.I)


def style_rows(messages, facts, k=1500, seed=0):
    """
    Your own past messages (from your Claude export), labelled as conversation.
    Teaches Topshe how YOU phrase things (Hinglish, shorthand, typos) so it doesn't mistake
    your normal talk for a phone command. Messages that look like phone commands are skipped.
    """
    rnd = random.Random(seed)
    pool = [m["text"] for m in messages
            if m.get("text") and not COMMAND_START.search(m["text"].strip()) and not COMMAND_ANYWHERE.search(m["text"])]
    rnd.shuffle(pool)
    out = []
    for t in pool[:k]:
        sys_prompt = with_facts(CMD_SYSTEM, facts if rnd.random() < 0.5 else [])
        out.append(row(sys_prompt, t, dumps({"action": "chat", "query": t}), "style"))
    return out




In [ ]:
# 4. Inputs from the topshe-data dataset (all optional)
def find(name):
    return next(iter(glob.glob(f"/kaggle/input/**/{name}", recursive=True)), None)

def read_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()] if path else []

interactions = read_jsonl(find("user_interactions.jsonl"))
facts = json.load(open(find("facts.json"))).get("facts", []) if find("facts.json") else []
past_messages = read_jsonl(find("claude_messages.jsonl"))
distill_cached = [r for r in read_jsonl(find("chat_distill.jsonl")) if r.get("base_model") == BASE_MODEL]
prev_eval = json.load(open(find("prev_eval.json"))) if find("prev_eval.json") else {}
print(f"interactions {len(interactions)} | facts {len(facts)} | past messages {len(past_messages)} | "
      f"cached chat {len(distill_cached)} | last week's scores: {bool(prev_eval)}")


In [ ]:
# 5. Build the dataset (with automatic re-weighting)
seed = int(time.time()) % 100000
synthetic = make_rows(seed=seed)                      # fresh phrasings every week
eval_synth = [r for r in synthetic if r["kind"] == "command"][:200]
eval_keys = {r["user"] for r in eval_synth}
train = [r for r in synthetic if r["user"] not in eval_keys]

# Re-weighting 1: action types the current brain got wrong more often get extra examples
per_action = ((prev_eval.get("synthetic") or {}).get("per_action") or {})
extra = []
for act, acc in per_action.items():
    copies = 2 if acc < 0.85 else (1 if acc < 0.95 else 0)
    pool = [r for r in train if r["kind"] == "command" and json.loads(r["assistant"]).get("action") == act]
    extra += pool * copies
    if copies:
        print(f"re-weighting: {act} was {acc:.0%} correct -> {copies} extra copies")
train += extra

# Re-weighting 2: your own commands (corrections 5x, confirmed 3x, ok 2x, recent 1.5x)
mine = user_rows(interactions, facts)
def held(r): return int(hashlib.md5(r["user"].encode()).hexdigest(), 16) % 100 < 15   # stable 15% for testing
eval_user = [r for r in mine if held(r)]
for r in mine:
    if not held(r):
        train += [r] * r["weight"]

# Your past messages (your words only): how you talk, so normal talk isn't mistaken for a command
style = style_rows(past_messages, facts, k=N_STYLE + 100, seed=seed)
eval_style, style = style[:100], style[100:]
train += style

eval_mem_chat, eval_mem_cmd = eval_memory_rows()
print("synthetic", len(synthetic), "| re-weighted extra", len(extra), "| yours", len(mine),
      "| style", len(style), "| eval: synth", len(eval_synth), "yours", len(eval_user), "style", len(eval_style))


In [ ]:
# 6. Load Qwen 3.5 4B (16-bit LoRA, language layers only)
from unsloth import FastModel
import torch

model, processor = FastModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_LEN,
    load_in_4bit=False, load_in_16bit=True, full_finetuning=False,
)
model = FastModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    use_gradient_checkpointing="unsloth", random_state=3407,
)
tok = getattr(processor, "tokenizer", processor)   # the text tokenizer inside the processor


In [ ]:
# 7. ONE prompt format for training AND the phone (saved as prompt_template.json)
def render_prompt(system, user):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    last = None
    for obj in (tok, processor):
        for extra in ({"enable_thinking": False}, {}):
            try:
                out = obj.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **extra)
                if isinstance(out, str) and ("{USER}" in out or user != "{USER}"):
                    return out
            except Exception as e:
                last = e
    raise RuntimeError(f"No chat template found: {last}")

TEMPLATE = render_prompt("{SYSTEM}", "{USER}")
assert TEMPLATE.count("{SYSTEM}") == 1 and TEMPLATE.count("{USER}") == 1, TEMPLATE
END = "<|im_end|>" if "<|im_end|>" in TEMPLATE else (tok.eos_token or "")
json.dump({"template": TEMPLATE, "end": END, "base_model": BASE_MODEL,
           "cmd_system": CMD_SYSTEM, "chat_system": CHAT_SYSTEM, "facts_header": FACTS_HEADER},
          open(f"{OUT}/prompt_template.json", "w"), ensure_ascii=False, indent=1)
print(repr(TEMPLATE))
tok.padding_side = "right"

def prompt_ids(system, user):
    return tok(TEMPLATE.replace("{SYSTEM}", system).replace("{USER}", user), add_special_tokens=False)["input_ids"]

def generate(system, user, max_new=96, model_=None):
    m = model_ or model
    ids = torch.tensor([prompt_ids(system, user)], device="cuda")
    out = m.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new,
                     do_sample=False, pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return text.split("</think>")[-1].strip()


In [ ]:
# 8. General chat, written by the base model itself (keeps its knowledge; made once, reused)
# LoRA starts as a no-op, so before training the model answers exactly like plain Qwen 3.5 4B.
if len(distill_cached) >= N_DISTILL // 2:
    distill = distill_cached
    print("Reusing", len(distill), "cached chat examples")
    shutil.copy(find("chat_distill.jsonl"), f"{OUT}/chat_distill.jsonl")
else:
    from datasets import load_dataset
    rnd = random.Random(1)
    alpaca = [r["instruction"] for r in load_dataset("yahma/alpaca-cleaned", split="train")
              if not r["input"] and 10 <= len(r["instruction"]) <= 200]
    rnd.shuffle(alpaca)
    yours = [m["text"] for m in past_messages if m["text"].rstrip().endswith("?") and 15 <= len(m["text"]) <= 200
             and not COMMAND_ANYWHERE.search(m["text"])]
    rnd.shuffle(yours)
    questions = alpaca[: N_DISTILL * 2 // 3] + yours[: N_DISTILL // 3]
    FastModel.for_inference(model)
    distill = []
    for i, q in enumerate(questions):
        a = generate(CHAT_SYSTEM, q, max_new=110)
        if 5 <= len(a) <= 500 and "<think>" not in a and not a.startswith("{"):
            distill.append({"question": q, "answer": a, "base_model": BASE_MODEL})
        if i % 100 == 0:
            print(i, "|", q[:60], "->", a[:80])
    FastModel.for_training(model)
    with open(f"{OUT}/chat_distill.jsonl", "w", encoding="utf-8") as f:
        for d in distill:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")
    print("Made", len(distill), "chat examples")
for d in distill:
    sys_prompt = with_facts(CHAT_SYSTEM, random_facts(4) if random.random() < 0.3 else [])
    train.append(row(sys_prompt, d["question"], d["answer"], "chat"))
random.shuffle(train)
counts = {k: sum(1 for r in train if r["kind"] == k) for k in sorted({r["kind"] for r in train})}
print("train rows:", len(train), counts)


In [ ]:
# 9. Tokenise (learn only Topshe's replies, not the prompts)
from datasets import Dataset

def encode(r):
    p_ids = prompt_ids(r["system"], r["user"])
    a_ids = tok(r["assistant"] + END + "\n", add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

ds = Dataset.from_list([encode(r) for r in train])
lens = sorted(len(x) for x in ds["input_ids"])
print(ds, "| tokens: median", lens[len(lens) // 2], "max", lens[-1])


In [ ]:
# 10. Train (stops safely if it runs too long)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, TrainerCallback

class TimeLimit(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if time.time() - T0 > TRAIN_HOURS_LIMIT * 3600:
            print("Time limit reached - stopping training early")
            control.should_training_stop = True

bf16 = torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=f"{TMP}/ckpt", per_device_train_batch_size=2, gradient_accumulation_steps=8,
    num_train_epochs=1, learning_rate=2e-4, warmup_steps=20, lr_scheduler_type="linear",
    logging_steps=25, optim="adamw_8bit", weight_decay=0.01, seed=3407,
    fp16=not bf16, bf16=bf16, save_strategy="no", report_to="none", remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
                  callbacks=[TimeLimit()])
trainer.train()
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print("loss: first", losses[:3], "last", losses[-3:])
assert losses and all(math.isfinite(x) for x in losses), "Training loss became NaN/inf - stopping (nothing will be published)"
model.save_pretrained(f"{OUT}/lora"); processor.save_pretrained(f"{OUT}/lora")
print("Training done in %.0f min" % ((time.time() - T0) / 60))


In [ ]:
# 11. Test (the weekly quality gate uses these numbers)
FastModel.for_inference(model)

def score(rows):
    valid = action = exact = 0
    per, misses = {}, []
    for r in rows:
        truth = json.loads(r["assistant"])
        raw = generate(r["system"], r["user"])
        ok_json, same = False, False
        try:
            pred = json.loads(raw[raw.index("{"): raw.rindex("}") + 1]); ok_json = True
            action += pred.get("action") == truth.get("action")
            same = pred == truth or (truth.get("action") == "chat" and pred.get("action") == "chat")
        except Exception:
            pass
        valid += ok_json; exact += same
        a = truth.get("action"); c = per.setdefault(a, [0, 0]); c[0] += same; c[1] += 1
        if not same: misses.append((r["user"], raw))
    n = max(len(rows), 1)
    return {"n": len(rows), "valid": valid / n, "action": action / n, "exact": exact / n,
            "per_action": {k: v[0] / v[1] for k, v in per.items()}, "misses": misses[:25]}

ev = {"synthetic": score(eval_synth)}
ev["yours"] = score(eval_user) if eval_user else None
ev["style_routing"] = score(eval_style)["action"] if eval_style else None      # your normal talk -> chat
ev["memory_commands"] = score(eval_mem_cmd)["exact"]
hits = 0
for r in eval_mem_chat:
    hits += r["assistant"].lower() in generate(r["system"], r["user"], 60).lower()
ev["memory_chat"] = hits / len(eval_mem_chat)
samples = {q: generate(with_facts(CHAT_SYSTEM, facts), q, 80) for q in
           ["who directed pather panchali?", "what is an instrumental variable, simply?", "who are you?"]}
ev["chat_samples"] = samples
ev["chat_ok"] = all(5 <= len(a) <= 600 and not a.startswith("{") for a in samples.values())
ev.update({"train_rows": len(train), "your_interactions": len(interactions), "base_model": BASE_MODEL,
           "loss_last": losses[-1], "minutes_to_eval": round((time.time() - T0) / 60)})
json.dump(ev, open(f"{OUT}/eval.json", "w"), ensure_ascii=False, indent=1)
print(json.dumps({k: (v if not isinstance(v, dict) or "n" not in v else {x: v[x] for x in ("n", "valid", "action", "exact")})
                  for k, v in ev.items()}, indent=1, ensure_ascii=False))


In [ ]:
# 12. Export for the phone: merge -> GGUF (same llama.cpp as the app) -> 4-bit -> smoke test
model.save_pretrained_merged(f"{TMP}/merged", processor, save_method="merged_16bit")
del model, trainer; torch.cuda.empty_cache()

LC = f"{TMP}/llama.cpp"
sh(f"git clone --depth 1 --branch {LLAMA_CPP_TAG} https://github.com/ggml-org/llama.cpp {LC}")
# The tokenizer isn't changed by training: use the base model's original tokenizer files
# (the converter's pinned transformers version is tested against those)
from huggingface_hub import snapshot_download
tdir = snapshot_download(BASE_MODEL, allow_patterns=["tokenizer*", "vocab*", "merges*", "special_tokens_map.json",
                                                     "added_tokens.json", "chat_template*"])
for f in glob.glob(f"{tdir}/*"):
    shutil.copy(f, f"{TMP}/merged/")
# The converter pins its own transformers/torch versions: keep them out of this environment
sh(f"python -m venv {TMP}/cvenv || (pip install -q virtualenv && python -m virtualenv {TMP}/cvenv)")
sh(f"{TMP}/cvenv/bin/pip install -q -r {LC}/requirements/requirements-convert_hf_to_gguf.txt")
sh(f"{TMP}/cvenv/bin/python {LC}/convert_hf_to_gguf.py {TMP}/merged --outfile {TMP}/brain-f16.gguf --outtype f16 --no-mtp")
shutil.rmtree(f"{TMP}/merged", ignore_errors=True)
sh(f"cmake -S {LC} -B {LC}/build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF -DLLAMA_BUILD_TESTS=OFF "
   f"-DLLAMA_BUILD_EXAMPLES=ON -DLLAMA_CURL=OFF > /dev/null")
sh(f"cmake --build {LC}/build --target llama-quantize llama-simple -j 4 > /dev/null")
sh(f"{LC}/build/bin/llama-quantize {TMP}/brain-f16.gguf {OUT}/topshe_brain.gguf Q4_K_M")
os.remove(f"{TMP}/brain-f16.gguf")

# Smoke test the exact file the phone will run, with the exact prompt format
smoke = [("wake me up at 6:45 tomorrow", "set_alarm"), ("play tum hi ho", "media"), ("call maa", "call"),
         ("remember that my gym is at 7", "remember"), ("go to sleep", "stop"), ("why is the sky blue", "chat")]
ok = 0
for text, want in smoke:
    p = TEMPLATE.replace("{SYSTEM}", CMD_SYSTEM).replace("{USER}", text)
    r = subprocess.run([f"{LC}/build/bin/llama-simple", "-m", f"{OUT}/topshe_brain.gguf", "-n", "60", p],
                       capture_output=True, text=True)
    gen = r.stdout.split("</think>")[-1]
    try:
        got = json.loads(gen[gen.index("{"): gen.index("}", gen.index("{")) + 1]).get("action")
    except Exception:
        got = None
    ok += got == want
    print(f"{text!r:35} -> {got}")
size = os.path.getsize(f"{OUT}/topshe_brain.gguf")
sha = hashlib.sha256(open(f"{OUT}/topshe_brain.gguf", "rb").read()).hexdigest()
ev.update({"gguf_smoke": ok / len(smoke), "gguf_bytes": size, "gguf_sha256": sha, "llama_cpp": LLAMA_CPP_TAG,
           "minutes_total": round((time.time() - T0) / 60)})
json.dump(ev, open(f"{OUT}/eval.json", "w"), ensure_ascii=False, indent=1)
print("topshe_brain.gguf: %.2f GB | phone-file smoke test %d/%d" % (size / 1e9, ok, len(smoke)))


In [ ]:
# 13. Done
for p in sorted(glob.glob(f"{OUT}/*")):
    print(os.path.basename(p))
print("Total %.0f min" % ((time.time() - T0) / 60))
